In [8]:
# 필수 라이브러리 설치
#!pip install langchain openai
from dotenv import load_dotenv
import os

# .env 파일을 불러와서 환경 변수로 설정
load_dotenv(dotenv_path='../.env')

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
print(OPENAI_API_KEY[:2])
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain.output_parsers import StructuredOutputParser, ResponseSchema

from pprint import pprint

# 출력 구조 정의 (평점, 장점, 단점, 요약)
response_schemas = [
    ResponseSchema(name="destination", description="여행지(도시 또는 지역명)"),
    ResponseSchema(name="duration", description="여행 기간(예: 2박 3일)"),
    ResponseSchema(name="budget", description="여행 예산(예: 30만원)"),
    ResponseSchema(name="rating", description="추천도(1~5점, 숫자만)"),
    ResponseSchema(name="activities", description="주요 활동 리스트(예: [해운대 바다구경, 자갈치시장 회 먹기])")
]
# 파서 초기화
parser = StructuredOutputParser.from_response_schemas(response_schemas)
format_instructions = parser.get_format_instructions()

print("출력 형식 지시사항:")
print(format_instructions)
# 프롬프트 템플릿
template = """
여행 후기나 계획 텍스트를 분석하세요. 텍스트: {review}

{format_instructions}
"""

prompt = ChatPromptTemplate.from_template(template)
prompt = prompt.partial(format_instructions=format_instructions)
# 모델 초기화 (temperature=0.5로 설정해 일관성 있는 출력)
#model = ChatOpenAI(temperature=0.7, model="gpt-3.5-turbo")
model = ChatOpenAI(
    #api_key=OPENAI_API_KEY,
    base_url="https://api.groq.com/openai/v1",  # Groq API 엔드포인트
    model="moonshotai/kimi-k2-instruct-0905",  # Spring AI와 동일한 모델
    temperature=0.7
)
# 테스트 리뷰 데이터
review = input('여행 후기나 계획을 입력하세요: ')
# 체인 실행
chain = prompt | model | parser

output = chain.invoke({"review": review})
# 결과 출력 (Pretty Print)
print("===== 분석 결과 =====")
pprint(output)

gs
출력 형식 지시사항:
The output should be a markdown code snippet formatted in the following schema, including the leading and trailing "```json" and "```":

```json
{
	"destination": string  // 여행지(도시 또는 지역명)
	"duration": string  // 여행 기간(예: 2박 3일)
	"budget": string  // 여행 예산(예: 30만원)
	"rating": string  // 추천도(1~5점, 숫자만)
	"activities": string  // 주요 활동 리스트(예: [해운대 바다구경, 자갈치시장 회 먹기])
}
```
===== 분석 결과 =====
{'activities': '[해운대 바다구경, 자갈치시장 회 먹기, 감천문화마을 구경]',
 'budget': '30만원',
 'destination': '부산',
 'duration': '2박 3일',
 'rating': '4'}
